In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185, ?

<h1>Monk Datasets SVM</h1>
<p>Exploring the three monk datasets with SVM Classifier.</p>
<hr/>

In [ ]:
import importlib
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict
import monk_common as mc
import svm_common as sc

importlib.reload(mc)

<h2>Exploring Datasets</h2>
<hr/>

In [ ]:
# Note: The regularization parameter C is explored on a logarithmic scale over four orders of magnitude,
# from strong (0.1) to weak regularization (100), following standard practice for SVM hyperparameter tuning.
param_grid_default = [
    {  # linear
        "svm__kernel": ["linear"],
        "svm__C": [0.1, 1, 10, 100],
    },
    {  # poly
        "svm__kernel": ["poly"],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale", 0.1, 1],
        "svm__degree": [2, 3, 4],
    },
    {  # sigmoid
        "svm__kernel": ["sigmoid"],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale", 0.1, 1]
    },
    {  # rbf
        "svm__kernel": ["rbf"],
        "svm__C": [0.1, 1, 10, 100],
        "svm__gamma": ["scale", 0.1, 1]
    },
]

# We only test SVC
pipe_default = Pipeline(steps=[
    ("svm", SVC())
])

# CV params
n_split = 5

# Of all parameters 'shuffle' is relevant because the TR set is small
# We also prefer StratifiedKFold as it preserves class balance during folding (optimal in small dataset)
cv_default = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=42)


<h3>Monk 1</h3>

<h4>Data Loading</h4>
<p>Load and prepare data</p>

In [ ]:
monk_1_TR_orig, monk_1_TS_orig = mc.load_set(1)
monk_1_TR_features, monk_1_TR_labels = mc.split_and_prepare_dataset(monk_1_TR_orig)
monk_1_TS_features, monk_1_TS_labels = mc.split_and_prepare_dataset(monk_1_TS_orig)
mc.dataset_introspection(monk_1_TR_orig, monk_1_TS_orig)

<h4>Model Selection</h4>
<p>Finding the best model with GridSearchCV.</p>

In [ ]:
scoring_1 = "accuracy"

# Perform a grid search
grid_1 = GridSearchCV(
    estimator = pipe_default,
    param_grid = param_grid_default,
    cv = cv_default,
    scoring = scoring_1,
    n_jobs = -1 # use all available cores
)

# Fit TR features and labels into grid
grid_1.fit(monk_1_TR_features, monk_1_TR_labels)
best_model_1 = grid_1.best_estimator_
sc.grid_introspection(grid_1)


In [ ]:
sc.plot_results_svc(grid_1.cv_results_)

<h4>Model Assesment</h4>
<p>Collecting various metrics for model assesment.</p>

In [ ]:
cv_predictions_1 = cross_val_predict(
    best_model_1,
    monk_1_TR_features, monk_1_TR_labels,
    cv=cv_default
)

print(classification_report(monk_1_TR_labels, cv_predictions_1, zero_division=0))

<h5>Training Accuracy Comparison</h5>
<p>The selected model is evaluated by comparing its training accuracy with the mean and standard deviation of the cross-validation accuracy. This comparison provides insights into the generalization capability of the model and helps identify potential overfitting or underfitting phenomena.</p>

In [ ]:
# Train score
train_score = best_model_1.score(monk_1_TR_features, monk_1_TR_labels)

# CV score
cv_scores = cross_val_score(
    best_model_1, monk_1_TR_features, monk_1_TR_labels,
    cv=cv_default, scoring=scoring_1
)

print(f"Train: {train_score:.3f}")
print(f"CV:    {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


<h5>Stability of cross-validation estimates</h5>
<p>To assess the robustness of the selected model, cross-validation was repeated using different random seeds for the data shuffling process. This allows evaluating the sensitivity of the estimated performance to different data partitions and provides an indication of the stability of the model under varying training/validation splits.</p>

In [ ]:
params = cv_default.__dict__.copy()
for seed in [0, 1, 2, 3, 4]:
    params["random_state"] = seed
    cv = cv_default.__class__(**params)
    scores = cross_val_score(
        best_model_1,
        monk_1_TR_features, monk_1_TR_labels,
        cv=cv,
        scoring=scoring_1,
    )
    print(cv, seed, scores.mean())

<h5>Features importance</h5>
<p>Computing features importance. Based on the rule used to build dataset <code>(a1 = a2) or (a5 = 1)</code>
we expect predominance of a1, a2 and a5.
</p>

In [ ]:
sv_freq, perm_imp, comp_attr = sc.compare_svfreq_vs_permutation(
    best_model_1, monk_1_TR_features, monk_1_TR_labels, scoring="accuracy"
)

print(comp_attr)

<h5>Bias Variance Introspection</h5>
<p>Monitor bias - variance. We expect that a high error impacts on performance, which are measured in accuracy.</p>
$$\mathbb{E}_{P}[(y - h(x))^2] = \text{Bias}^2 + \text{Variance} + \text{Noise}$$
<p>
<ul>
<li>Bias: the train accuracy is stable to 1 independently from the data size. The model is able to interpolate the function even with little data. The model has therefore very low bias.</li>
<li>Variance: The cross-validation accuracy shows a consistent gap with respect to the training accuracy for small training set sizes, indicating a high-variance regime. However, this gap progressively decreases as more data are used, and variance is effectively resolved for larger training sets.</li>
<li>Noise: monk 1 dataset is known to be noise free</li>
</ul>

In [ ]:
sc.plot_learning_curve_svm(best_model_1, monk_1_TR_features, monk_1_TR_labels, cv_default)

<h4>Test predictions</h4>

In [ ]:
best_model_1 = grid_1.best_estimator_

test_predictions_1 = best_model_1.predict(monk_1_TS_features)
test_accuracy_1 = best_model_1.score(monk_1_TS_features, monk_1_TS_labels)

print(classification_report(monk_1_TS_labels, test_predictions_1, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(monk_1_TS_labels, test_predictions_1, xticks_rotation='vertical', cmap='Blues');

<hr/>

<h3>Monk 2</h3>

<h4>Data Loading</h4>
<p>Load and prepare data</p>

In [ ]:
monk_2_TR_orig, monk_2_TS_orig = mc.load_set(2)
monk_2_TR_features, monk_2_TR_labels = mc.split_and_prepare_dataset(monk_2_TR_orig)
monk_2_TS_features, monk_2_TS_labels = mc.split_and_prepare_dataset(monk_2_TS_orig)
mc.dataset_introspection(monk_2_TR_orig, monk_2_TS_orig)

<h4>Model Selection</h4>
<p>Finding the best model with GridSearchCV.</p>

In [ ]:
scoring_2 = "balanced_accuracy" # <-- because the training set shows an unbalanced scenario
grid_2 = GridSearchCV(
    estimator = pipe_default,
    param_grid = param_grid_default,
    cv = cv_default,
    scoring = scoring_2,
)

#warnings.filterwarnings("ignore", category=ConvergenceWarning)
grid_2.fit(monk_2_TR_features, monk_2_TR_labels)
best_model_2 = grid_2.best_estimator_
sc.grid_introspection(grid_2)

In [ ]:
sc.plot_results_svc(grid_2.cv_results_)

<h4>Model Assesment</h4>
<p>Collecting various metrics for model assesment.</p>

In [ ]:
cv_predictions_2 = cross_val_predict(
    best_model_2,
    monk_2_TR_features, monk_2_TR_labels,
    cv=cv_default
)

print(classification_report(monk_2_TR_labels, cv_predictions_2, zero_division=0))

<h5>Training Accuracy Comparison</h5>
<p>The selected model is evaluated by comparing its training accuracy with the mean and standard deviation of the cross-validation accuracy. This comparison provides insights into the generalization capability of the model and helps identify potential overfitting or underfitting phenomena.</p>

In [ ]:
# Train score
train_score = best_model_2.score(monk_2_TR_features, monk_2_TR_labels)

# CV score
cv_scores = cross_val_score(
    best_model_2, monk_2_TR_features, monk_2_TR_labels,
    cv=cv_default, scoring=scoring_2
)

print(f"Train: {train_score:.3f}")
print(f"CV:    {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


<h5>Stability of cross-validation estimates</h5>
<p>To assess the robustness of the selected model, cross-validation was repeated using different random seeds for the data shuffling process. This allows evaluating the sensitivity of the estimated performance to different data partitions and provides an indication of the stability of the model under varying training/validation splits.</p>

In [ ]:
params = cv_default.__dict__.copy()
for seed in range(1,6):
    params["random_state"] = seed
    cv = cv_default.__class__(**params)
    scores = cross_val_score(
        best_model_2,
        monk_2_TR_features, monk_2_TR_labels,
        cv=cv,
        scoring=scoring_2
    )
    print(cv, seed, scores.mean())

<h5>Features importance</h5>
<p>Computing features importance. Based on the rule used to build dataset <code>(EXACTLY TWO of {a1 = 1, a2 = 1, a3 = 1, a4 = 1, a5 = 1, a6 = 1}</code>
we expect an almost equal distribution of predominance of all attributes.
</p>

In [ ]:
sv_freq, perm_imp, comp_attr = sc.compare_svfreq_vs_permutation(
    best_model_2, monk_2_TR_features, monk_2_TR_labels, scoring="accuracy"
)

print(comp_attr)

<h5>Bias Variance Introspection</h5>
<p>Monitor bias - variance. We expect that a high error impacts on performance, which are measured in accuracy.</p>
$$\mathbb{E}_{P}[(y - h(x))^2] = \text{Bias}^2 + \text{Variance} + \text{Noise}$$
<p>
<ul>
<li>Bias: the train accuracy is stable to 1 independently from the data size. The model is able to interpolate the function even with little data. The model has therefore very low bias.</li>
<li>Variance: the cv shows a consistent gap when compared with train curve. The gap is however progressively reduced by using more data, the model is therefore subjected to high variance. Also, the fact that the gap is never closed means that the model suffers from an amount of variance.</li>
<li>Noise: monk 1 dataset is known to be noise free</li>
</ul>

In [ ]:
sc.plot_learning_curve_svm(best_model_2, monk_2_TR_features, monk_2_TR_labels, cv_default, scoring=scoring_2)

<h4>Test predictions</h4>

In [ ]:
test_predictions_2 = best_model_2.predict(monk_2_TS_features)
test_accuracy_2 = best_model_2.score(monk_2_TS_features, monk_2_TS_labels)

In [ ]:
print(classification_report(monk_2_TS_labels, test_predictions_2, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(monk_2_TS_labels, test_predictions_2, xticks_rotation='vertical', cmap='Blues');

<hr/>

<h3>Monk 3</h3>

<h4>Data Loading</h4>
<p>Load and prepare data</p>

In [ ]:
monk_3_TR_orig, monk_3_TS_orig = mc.load_set(3)
monk_3_TR_features, monk_3_TR_labels = mc.split_and_prepare_dataset(monk_3_TR_orig)
monk_3_TS_features, monk_3_TS_labels = mc.split_and_prepare_dataset(monk_3_TS_orig)
mc.dataset_introspection(monk_3_TR_orig, monk_3_TS_orig)

In [ ]:
scoring_3 = "accuracy"

grid_3 = GridSearchCV(
    estimator = pipe_default,
    param_grid = param_grid_default,
    cv = cv_default,
    scoring = scoring_3,
)

#warnings.filterwarnings("ignore", category=ConvergenceWarning)
grid_3.fit(monk_3_TR_features, monk_3_TR_labels)
best_model_3 = grid_3.best_estimator_

sc.grid_introspection(grid_3)

In [ ]:
sc.plot_results_svc(grid_3.cv_results_)

<p>Computing features importance. Based on the rule used to build dataset <code>(a5 = 3 and a4 = 1) or (a5 = 4 and a2 = 3)</code>
we expect predominance of a5, a4, a2
</p>

In [ ]:
sv_freq, perm_imp, comp_attr = sc.compare_svfreq_vs_permutation(
    best_model_3, monk_3_TR_features, monk_3_TR_labels, scoring=scoring_3
)

print("Frequency vs importance (a1..a6):")
print(comp_attr)

<h4>Model Assesment</h4>
<p>Collecting various metrics for model assesment.</p>

In [ ]:
cv_predictions_3 = cross_val_predict(
    best_model_3,
    monk_3_TR_features, monk_3_TR_labels,
    cv=cv_default
)

print(classification_report(monk_3_TR_labels, cv_predictions_3, zero_division=0))

<h5>Training Accuracy Comparison</h5>
<p>The selected model is evaluated by comparing its training accuracy with the mean and standard deviation of the cross-validation accuracy. This comparison provides insights into the generalization capability of the model and helps identify potential overfitting or underfitting phenomena.</p>

In [ ]:
# Train score
train_score_3 = best_model_3.score(monk_3_TR_features, monk_3_TR_labels)

# CV score (stessa CV della GridSearch!)
cv_scores_3 = cross_val_score(
    best_model_3, monk_3_TR_features, monk_3_TR_labels,
    cv=cv_default, scoring=scoring_3
)

print(f"Train: {train_score_3:.3f}")
print(f"CV:    {cv_scores_3.mean():.3f} ± {cv_scores_3.std():.3f}")


<h5>Stability of cross-validation estimates</h5>
<p>To assess the robustness of the selected model, cross-validation was repeated using different random seeds for the data shuffling process. This allows evaluating the sensitivity of the estimated performance to different data partitions and provides an indication of the stability of the model under varying training/validation splits.</p>

In [ ]:
params = cv_default.__dict__.copy()
for seed in range(1,6):
    params["random_state"] = seed
    cv = cv_default.__class__(**params)
    scores = cross_val_score(
        best_model_3,
        monk_3_TR_features, monk_3_TR_labels,
        cv=cv,
        scoring=scoring_3
    )
    print(cv, seed, scores.mean())

<h5>Bias Variance Introspection</h5>
<p>Applying various techniques to monitor bias - variance.</p>
$$\mathbb{E}_{P}[(y - h(x))^2] = \text{Bias}^2 + \text{Variance} + \text{Noise}$$
<p>
<ul>
<li>Bias: the train accuracy is stable to 1 independent from the data size so the model has bias 0</li>
<li>Variance: the cv shows an increase in accuracy with more data, the model is therefore subjected to high variance</li>
<li>Noise: monk 3 dataset is known to have noise</li>
</ul>

In [ ]:
sc.plot_learning_curve_svm(best_model_3, monk_3_TR_features, monk_3_TR_labels, cv_default)

<h5>Features importance</h5>
<p>Computing features importance. Based on the rule used to build dataset <code>(a5 = 3 and a4 = 1) or (a5 = 4 and a2 = 3)</code>
we expect an almost equal distribution of predominance of a5, a4 and a2 attributes.
</p>

In [ ]:
sv_freq, perm_imp, comp_attr = sc.compare_svfreq_vs_permutation(
    best_model_3, monk_3_TR_features, monk_3_TR_labels, scoring="accuracy"
)

print(comp_attr)

<h4>Test predictions</h4>

In [ ]:
test_predictions_3 = best_model_3.predict(monk_3_TS_features)
test_accuracy_3 = best_model_3.score(monk_3_TS_features, monk_3_TS_labels)

In [ ]:
print(classification_report(monk_3_TS_labels, test_predictions_3, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(monk_3_TS_labels, test_predictions_3, xticks_rotation='vertical', cmap='Blues');